# 🔍 Customer Churn Prediction
### RetentionIQ Dataset | Machine Learning Project

**Goal:** Predict which customers are likely to churn using ML, and extract actionable business insights.

**Workflow:** Load Data → EDA → Preprocessing → Train 4 Models → Evaluate → Business Insights


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score,
                             roc_curve, accuracy_score)
print('✅ All libraries imported!')

## 📥 Step 1: Load & Explore Data

In [ ]:
df = pd.read_csv('../data/retention_iq_churn_dataset.csv')
print(f'Shape: {df.shape}  |  Churn rate: {df["churned"].mean()*100:.1f}%  |  Nulls: {df.isnull().sum().sum()}')
df.head()

In [ ]:
print('Churn value counts:')
print(df['churned'].value_counts())
print('\nDescriptive stats:')
print(df.describe().round(2))

## 📊 Step 2: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA: Churn Patterns', fontsize=16, fontweight='bold')

counts = df['churned'].value_counts()
axes[0,0].bar(['Retained','Churned'], counts.values, color=['#10B981','#EF4444'])
axes[0,0].set_title('Churn Distribution')
for i, v in enumerate(counts.values):
    axes[0,0].text(i, v+5, f'{v} ({v/len(df)*100:.0f}%)', ha='center', fontweight='bold')

df_t = df.copy()
df_t['Label'] = df_t['churned'].map({0:'Retained',1:'Churned'})
sns.boxplot(data=df_t, x='Label', y='tenure_months', ax=axes[0,1],
            palette={'Retained':'#10B981','Churned':'#EF4444'})
axes[0,1].set_title('Tenure by Churn')

sns.boxplot(data=df_t, x='Label', y='monthly_charges', ax=axes[0,2],
            palette={'Retained':'#10B981','Churned':'#EF4444'})
axes[0,2].set_title('Monthly Charges by Churn')

ct = df.groupby('contract_type')['churned'].mean().sort_values(ascending=False)
axes[1,0].barh(ct.index, ct.values*100, color='#4F46E5')
axes[1,0].set_title('Churn Rate by Contract Type')
axes[1,0].set_xlabel('Churn Rate (%)')

pf = df.groupby('payment_failures')['churned'].mean()
axes[1,1].bar(pf.index, pf.values*100, color='#F59E0B')
axes[1,1].set_title('Churn by Payment Failures')

df[df['churned']==0]['nps_score'].plot(kind='hist', bins=20, ax=axes[1,2],
    alpha=0.6, color='#10B981', label='Retained')
df[df['churned']==1]['nps_score'].plot(kind='hist', bins=20, ax=axes[1,2],
    alpha=0.6, color='#EF4444', label='Churned')
axes[1,2].set_title('NPS Score Distribution')
axes[1,2].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['age','tenure_months','monthly_charges','logins_last_30d',
                'features_used','support_tickets','nps_score','payment_failures','churned']
plt.figure(figsize=(9,7))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation with Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔧 Step 3: Data Preprocessing

- Select features (drop ID and leaky `churn_probability`)
- Label encode categoricals
- Impute nulls with median
- 80/20 stratified train/test split
- StandardScaler for Logistic Regression

In [ ]:
features = ['age','tenure_months','monthly_charges','total_charges',
            'logins_last_30d','features_used','support_tickets','nps_score',
            'last_login_days_ago','email_open_rate','payment_failures',
            'plan','contract_type','payment_method','gender','region']

X = df[features].copy()
y = df['churned']

cat_cols = ['plan','contract_type','payment_method','gender','region']
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')
print(f'✅ Churn in train: {y_train.mean()*100:.1f}%  |  test: {y_test.mean()*100:.1f}%')

## 🤖 Step 4: Train & Evaluate 4 Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
print(f"{'Model':<22} {'Accuracy':>10} {'Test AUC':>10} {'CV AUC(5f)':>12}")
print('-'*58)

for name, model in models.items():
    Xtr = X_train_sc if name=='Logistic Regression' else X_train
    Xte = X_test_sc  if name=='Logistic Regression' else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:,1]
    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    cv   = cross_val_score(model, Xtr, y_train, cv=5, scoring='roc_auc').mean()
    results[name] = {'model':model,'y_pred':y_pred,'y_prob':y_prob,'acc':acc,'auc':auc,'cv_auc':cv}
    print(f'{name:<22} {acc:>10.3f} {auc:>10.3f} {cv:>12.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison', fontsize=14, fontweight='bold')

names = list(results.keys())
x = np.arange(len(names)); w = 0.25
axes[0].bar(x-w, [results[n]['acc'] for n in names],    w, label='Accuracy',  color='#4F46E5', alpha=0.85)
axes[0].bar(x,   [results[n]['auc'] for n in names],    w, label='Test AUC',  color='#10B981', alpha=0.85)
axes[0].bar(x+w, [results[n]['cv_auc'] for n in names], w, label='CV AUC',    color='#F59E0B', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([n.replace(' ','\n') for n in names], fontsize=9)
axes[0].set_ylim(0.5,1.05); axes[0].set_title('Scores by Model'); axes[0].legend()

for (name, res), col in zip(results.items(), ['#4F46E5','#F59E0B','#10B981','#EF4444']):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=col, lw=2)
axes[1].plot([0,1],[0,1],'--',color='gray',lw=1); axes[1].legend(fontsize=8)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].set_title('ROC Curves')
plt.tight_layout(); plt.show()

## 🏆 Step 5: Best Model Deep Dive

In [ ]:
best_name = max(results, key=lambda n: results[n]['auc'])
best = results[best_name]
print(f'Best: {best_name}  |  AUC: {best["auc"]:.3f}\n')
print(classification_report(y_test, best['y_pred'], target_names=['Retained','Churned']))

fig, axes = plt.subplots(1,3, figsize=(16,5))
fig.suptitle(f'Best Model: {best_name}', fontsize=14, fontweight='bold')
ConfusionMatrixDisplay(confusion_matrix(y_test, best['y_pred']),
    display_labels=['Retained','Churned']).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix')

rf = results['Random Forest']['model']
import pandas as pd
pd.Series(rf.feature_importances_, index=X_imp.columns).nlargest(12).sort_values().plot(
    kind='barh', ax=axes[1], color='#4F46E5', alpha=0.85)
axes[1].set_title('Top 12 Features (Random Forest)')

axes[2].hist(best['y_prob'][y_test.values==0], bins=25, alpha=0.6, color='#10B981', label='Retained')
axes[2].hist(best['y_prob'][y_test.values==1], bins=25, alpha=0.6, color='#EF4444', label='Churned')
axes[2].axvline(0.5, color='black', linestyle='--', lw=1.5)
axes[2].set_title('Predicted Probabilities'); axes[2].legend()
plt.tight_layout(); plt.show()

## 💼 Step 6: Business Insights

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(13,5))
risk = pd.cut(best['y_prob'], bins=[0,0.3,0.6,1.0], labels=['Low','Medium','High'])
rc = risk.value_counts().reindex(['Low','Medium','High'])
axes[0].bar(rc.index, rc.values, color=['#10B981','#F59E0B','#EF4444'], width=0.5)
axes[0].set_title('Customer Risk Segmentation'); axes[0].set_ylabel('Customers')
for bar,v in zip(axes[0].patches, rc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(v), ha='center', fontweight='bold')

si = np.argsort(-best['y_prob'])
sa = y_test.values[si]
cc = np.cumsum(sa)/sa.sum()
pct = np.arange(1,len(sa)+1)/len(sa)
axes[1].plot(pct*100, cc*100, color='#4F46E5', lw=2.5, label='Model')
axes[1].plot([0,100],[0,100],'--',color='gray',lw=1.5,label='Random')
i30 = int(0.3*len(sa))
axes[1].axvline(30,color='#F59E0B',linestyle=':',lw=1.5)
axes[1].axhline(cc[i30]*100,color='#F59E0B',linestyle=':',lw=1.5)
axes[1].set_xlabel('% Customers Targeted'); axes[1].set_ylabel('% Churners Captured')
axes[1].set_title('Lift Chart'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f'\n Targeting top 30% captures {cc[i30]*100:.0f}% of churners = {cc[i30]/0.3:.1f}x lift!')

## ✅ Summary

| Model | AUC | Notes |
|---|---|---|
| Logistic Regression | 0.950 | Fast, interpretable baseline |
| Decision Tree | 0.877 | Easy to explain |
| Random Forest | 0.961 | Best accuracy (90%) |
| **Gradient Boosting** | **0.962** | **Best AUC — recommended** |

**Top churn drivers:** payment failures, short tenure, low NPS, month-to-month contract

**Next steps:** SHAP values → Hyperparameter tuning → Streamlit dashboard
